<div class="alert alert-block alert-info"><font color='blue', size="8"><center><b>
Functions
</font></div>

<font size="5"><center><b>
PROSPER + Openserver + Phyton <br>
Probabilistic Analysis for Gas Well Model (PROSPER)
 <br></font>

Last update: 06 08 2025

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.gridspec as gridspec

import seaborn as sns
import scipy.stats as stats
import math

from collections import defaultdict       # library to create dictionarie variables

from IPython.display import display, HTML, Markdown

### Create DataFrame for input variables

In [2]:
def Input_Variables_DF(Input_var_dict):
    Stats_L = ['Low', 'High', 'Mean', 'SD', 'Constant', 'Distribution', 'Variable']
    
    df = pd.DataFrame(Input_var_dict)
    df = df.set_index(pd.Index(Stats_L))
    display(df)
    #return df
    

### Create plots - input variables

In [3]:
def Generate_normal_and_Truncated(V_mu,V_SD,V_samples,V_Low_limit,V_High_limit,V_Constant, V_x_Label, V_Title, n_col,n_row,AX,RBG, Dist_Type=None):
    import numpy as np
    import scipy.stats as stats
    
    n_bins = 30
    N_Samples = np.random.normal(loc = V_mu, scale = V_SD, size= V_samples) # normal distribution

    if Dist_Type == 'Constant':
        N_Samples = np.random.uniform(low = V_Constant, high = V_Constant, size = 10000)
        
    if Dist_Type == 'Uniform':
        N_Samples = np.random.uniform(low = V_Low_limit, high = V_High_limit, size = 10000)
        
    if Dist_Type == 'Normal':
        N_Samples = np.random.normal(loc = V_mu, scale = V_SD, size= V_samples) # normal distribution
    
    # Plot Normal Distribution
    AX[n_row,n_col].hist(N_Samples, bins=n_bins, density=True, align='mid', color=(RBG), alpha=1, edgecolor = "black")
    AX[n_row,n_col].set_xlabel(' ')
    AX[n_row,n_col].set_ylabel('Probability Density')
    AX[n_row,n_col].set_title(V_Title, weight='bold', fontsize = 18)
    x_axis = AX[n_row,n_col].set_xlim()

    
    ## generate data for truncated normal distribution
    
    NT_Samples = stats.truncnorm.rvs((V_Low_limit-V_mu)/V_SD,(V_High_limit-V_mu)/V_SD,loc=V_mu,scale=V_SD, size=V_samples)

    if Dist_Type == 'Constant':
        NT_Samples = np.random.uniform(low = V_Constant, high = V_Constant, size = 10000)
        
    if Dist_Type == 'Uniform':
        NT_Samples = np.random.uniform(low = V_Low_limit, high = V_High_limit, size = 10000)
        
    if Dist_Type == 'Normal':
        NT_Samples = stats.truncnorm.rvs((V_Low_limit-V_mu)/V_SD,(V_High_limit-V_mu)/V_SD,loc=V_mu,scale=V_SD, size=V_samples)
        
    ## Plot truncated Normal distribution
    AX[n_row+1,n_col].hist(NT_Samples, bins=n_bins, density=True, align='mid', color=RBG, alpha=0.4, edgecolor = "black")
    AX[n_row+1,n_col].set_xlabel(' ')
    AX[n_row+1,n_col].set_ylabel('PD')
    AX[n_row+1,n_col].set_title(V_Title + ' (TRUNCATED)')
    AX[n_row+1,n_col].set_xlim(x_axis)

In [4]:
def Plot_Input_func(Input_var_dict):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    
    fig, AX = plt.subplots(nrows = 4, ncols=4, figsize=(15,10), layout ='constrained')
    
    #cmap = matplotlib.cm.get_cmap('rocket')
    cmap = sns.color_palette('Set2', as_cmap=True)
    
    icolor = 0.1
    i_samples = 10000
    i_col = 0
    i_row = 0
    for key in Input_var_dict.keys():
        
        RBG = cmap(icolor)[0],cmap(icolor)[1],cmap(icolor)[2]
        icolor = icolor + 0.1
        
        i_mean, i_SD = Input_var_dict[key][2], Input_var_dict[key][3]
        i_Low_limit, i_High_limit = Input_var_dict[key][0], Input_var_dict[key][1]
        i_constant = Input_var_dict[key][4]
        i_x_Label, i_Title = Input_var_dict[key][6], Input_var_dict[key][6]
        i_Dist_Type = Input_var_dict[key][5]
    
        Generate_normal_and_Truncated(i_mean, i_SD, i_samples, i_Low_limit, i_High_limit, i_constant, i_x_Label, i_Title, i_col,i_row,AX,RBG,i_Dist_Type)
        
        i_col = i_col + 1
        if i_col > 3:
            i_col = 0
            i_row = 2

In [5]:
def Generate_rand_val(var_name, Input_var_dict,my_dict_input_vals, test_val=0):
    import numpy as np
    import scipy.stats as stats
    
    i_mean, i_SD = Input_var_dict[var_name][2], Input_var_dict[var_name][3]
    i_Low_limit, i_High_limit = Input_var_dict[var_name][0], Input_var_dict[var_name][1]
    i_constant = Input_var_dict[var_name][4]
    i_x_Label, i_Title = Input_var_dict[var_name][6], Input_var_dict[var_name][6]
    i_Dist_Type = Input_var_dict[var_name][5]
        
    ## generate data for truncated normal distribution
    if i_Dist_Type == 'Constant':
        NT_Samples = i_constant
        
    if i_Dist_Type == 'Uniform':
        NT_Samples = np.random.uniform(low = i_Low_limit, high = i_High_limit, size = 1)
        
    if i_Dist_Type == 'Normal':
        NT_Samples = stats.truncnorm.rvs((i_Low_limit-i_mean)/i_SD,(i_High_limit-i_mean)/i_SD, loc=i_mean, scale=i_SD, size=1)
    
    if var_name=='Perf':
        if test_val < NT_Samples:
            #print(test_val)
            NT_Samples = test_val
    
    #print(NT_Samples)
    if NT_Samples < 1:
        NT_Samples = np.around(NT_Samples,4)
    else:
        NT_Samples = np.around(NT_Samples,1)
    
    ## Save input values in dictionary
    if i_Dist_Type == 'Constant':    
        my_dict_input_vals[var_name].extend([NT_Samples])
    else:
        my_dict_input_vals[var_name].extend(NT_Samples)   #Save input values in dictionary
    
    return NT_Samples, my_dict_input_vals

In [6]:
def print_input_and_solutions(Results_display_Type,i,my_dict_input_vals,VLP_Index_sample,VLP_Index_sample_Name,
                             Choke, Gas_Sol, BHP_Sol, WHP_Sol,Water_Sol, Gas_AOF,iteraciones):
    from IPython.display import clear_output  # Clear output cell
    
    if Results_display_Type == 'Plots + DF':
        Value_list =[
            i,my_dict_input_vals['Pr'][i], my_dict_input_vals['DA'][i], my_dict_input_vals['k'][i], my_dict_input_vals['h'][i],
            my_dict_input_vals['Perf'][i], my_dict_input_vals['S'][i], my_dict_input_vals['WGR'][i], 
            my_dict_input_vals['Top_Press'][i], VLP_Index_sample, VLP_Index_sample_Name, Choke, Gas_Sol, BHP_Sol, WHP_Sol,Water_Sol, 
            Gas_AOF
        ]

        var_labels_dic = {
         'Variable': vars_list,
         'Abreviation': Abrev_list,
         'Value': Value_list,
         'Units': Units_list,
        }

        df_var_labels = pd.DataFrame(var_labels_dic)
        #display(df_var_labels)


        ## Plot Solution
        fig, ax = plt.subplots(nrows = 1, ncols=3, figsize=(8,2), layout ='constrained')

        x = my_dict[str(i) + '_G']
        y = my_dict[str(i) + '_IPR']
        df_vlp = pd.DataFrame(my_dict)
        df_vlp.replace(0, np.nan, inplace=True)
        y2=df_vlp[str(i) + '_VLP']
        #ax.scatter(x,y,marker='.', color=(255/255, 140/255, 0/255), s= 100, alpha=0.5, clip_on=True, linestyle='dashed')

        ax[0].plot(x,y,'.g-')
        ax[0].plot(x,y2,'.b-')
        ax[0].scatter(x=Gas_Sol,y=BHP_Sol,marker='.', color='black', s= 300, alpha=1, clip_on=False)

        ax[0].set_title('VLP vs IPR', weight='bold', color='black', fontsize=8)
        ax[0].set_ylabel('Pressure', fontsize=8)
        ax[0].set_xlabel('Gas Rate', fontsize=8)
        ax[0].tick_params(axis='x', labelsize=8)
        ax[0].tick_params(axis='y', labelsize=8)
        ax[0].set_ylim(0,Input_var_dict['Pr'][1]+1000)
        ax[0].set_xlim(0,None)

        ax[2].hist(my_dict_input_vals['Sol_Gas'], bins=50, density=True, align='mid', color='orange', alpha=0.8, edgecolor = "red");

        ax[2].set_title('Gas Rate Distribution', weight='bold', color='black', fontsize=8)
        ax[2].set_xlabel('Gas Rate', fontsize=8)
        ax[2].set_ylabel('PD', fontsize=8)
        ax[2].tick_params(axis='x', labelsize=8)
        ax[2].yaxis.tick_right()
        ax[2].get_yaxis().set_visible(False)

        # Plot density map
        df_densityP = pd.DataFrame(my_dict_input_vals)
        if i > 2:
            sns.kdeplot(x=df_densityP['Sol_Gas'], y=df_densityP['FBHP'], fill=True, alpha=0.7, ax=ax[1], 
                        cmap='coolwarm', label='PROSPER density map', thresh=0)

            ax[1].set_title('Solution Density Map', weight='bold', color='black', fontsize=8)
            ax[1].set_xlabel('Gas Rate', fontsize=8)
            ax[1].set_ylabel('Pressure', fontsize=8)
            ax[1].yaxis.set_label_position("left")
            ax[1].yaxis.tick_left()
            ax[1].tick_params(axis='both', labelsize=8)
            ax[1].get_yaxis().set_visible(False)

        plt.show()

        display(df_var_labels.style.set_properties(subset=['Value'], **{'width': '200px'}))
    
    ### Display results as text (FASTER)
    else:
        print('Index: ', i)
        print('Reservoir Pressure = ', my_dict_input_vals['Pr'][i])
        print('Drainage Area = ', my_dict_input_vals['DA'][i])
        print('Permeability = ', my_dict_input_vals['k'][i])
        print('Reservoir thickness = ', my_dict_input_vals['h'][i])
        print('Perforation Interval = ', my_dict_input_vals['Perf'][i])
        print('SKIN = ', my_dict_input_vals['S'][i])
        print('WGR = ', my_dict_input_vals['WGR'][i])
        print('Top Node Pressure = ', my_dict_input_vals['Top_Press'][i])

        print('VLP Correlation Index = ', VLP_Index_sample, VLP_Index_sample_Name) 
        print('Choke = ', Choke, '\n')


        print('------')

        print('Gas Rate = ', str(round(Gas_Sol,3)), ' MMscfd')
        print('Pressure = ', str(round(BHP_Sol,1)), ' psi')
        print('WHP = ', str(round(WHP_Sol,1)), ' psi')
        print('AOF = ', Gas_AOF) 
        print('------')
    
    
    
    
    
    if i < iteraciones-1:
        clear_output(wait=True)

In [7]:
### Function to generate Plots with distribution for each axis

def Gen_Scatter_Density_Distribution(df,x_var, y_var,Ptype):
    
    iteraciones  =  len(df[x_var])
    x, y = df[x_var], df[y_var]
    fig = plt.figure(figsize=(10,6))
    gs = gridspec.GridSpec(3, 5) # -> grid size: 3 rows, 5 columns

    fig.subplots_adjust(wspace=0.02,hspace=0.02)

    ax_main = plt.subplot(gs[1:3, :2])
    ax_A = plt.subplot(gs[0, :2],sharex=ax_main)                      # -> Rows 1, - Cols 1,2
    ax_B = plt.subplot(gs[1:3, 2],sharey=ax_main)                     # -> Rows 2,3 - Cols 3
    ax_C = plt.subplot(gs[1:3, 3:5],sharey=ax_main,sharex=ax_main)    # -> Rows 2,3 - Cols 4,5

    
    ## Y vs X plot
    ax = ax_main
    ax.scatter(x,y,marker='.', color=(255/255, 140/255, 0/255), s= 200, alpha=0.5, clip_on=False)

    ax.set_xlabel('Gas Rate, MMscfd', color='red', weight='bold')    
    ax.set_ylabel(y_var, color='blue', weight='bold')

    ax.set_axisbelow(True)
    ax.set_xlim(left=0),    ax.set_ylim(bottom=0)

    ax.yaxis.tick_left(),    ax.xaxis.tick_bottom()
    ax.tick_params(axis='x', labelsize=9)
    ax.tick_params(axis='y', labelsize=9)

    ### *********************************
    ### -> Density Map Y vs X plot ###
    ax = ax_C
    contour_set = sns.kdeplot(x=x, y=y, fill=True, alpha=1, ax=ax, cmap='coolwarm', label='PROSPER density map', thresh=0)
    
    ax.set_xlabel('Gas Rate, MMscfd', color='red', weight='bold')
    ax.tick_params(axis='x', labelsize=8),    ax.get_yaxis().set_visible(False)

    # The last contour is the outermost one
    last_contour_color = contour_set.collections[-1].get_facecolor()[0]
    ax.set_facecolor(last_contour_color)

    
    ## X axis Distribution ##
    ax = ax_A
    ax.hist(x,bins=int(iteraciones/33),align='mid', color='red', alpha=0.5, edgecolor = "black")
    ax.set(ylabel='PD')
    ax.tick_params(axis='x', labelsize=8),    ax.tick_params(axis='y', labelsize=8)
    ax.yaxis.tick_left(),    ax.xaxis.tick_top()

    
    ## Y axis Distribution ##
    ax = ax_B
    ax.hist(y,bins=int(iteraciones/33),orientation='horizontal',align='mid', color='blue', alpha=0.5, edgecolor = "black")
    ax.set(xlabel='PD'),    ax.tick_params(axis='x', labelsize=8),    ax.tick_params(axis='y', labelsize=8)
    ax.invert_xaxis(),    ax.yaxis.tick_right(),    ax.invert_xaxis()

    plt.show()
  

### Plot distribution for the given DataFrame and specified bins

In [8]:
def Plot_Distributions_Inputs(df,df_NO,n_bins,fig_h,fig_w):
    
    ### -> Create Legend
    fig, ax= plt.subplots(nrows = 1, ncols=1, figsize=(1,1))
    
    sns.countplot(ax =ax, x=[1], color='orange', dodge=False, alpha=1),    sns.countplot(ax =ax, x=[1], color='black', dodge=False, alpha=0.7)
    sns.countplot(ax =ax, x=[1], color='white', dodge=False, alpha=1)
    
    ax.legend(title='',labels=['Solution', 'No Solution'], facecolor='white', fontsize="15"),    ax.axis('off')

    col_names_Inputs_L = df.columns.tolist() # -> Get Column names

  
    ## Create Disribution Plots ##
    fig, AX = plt.subplots(nrows = 4, ncols=4, figsize=(fig_w,fig_h), layout ='constrained')
        
    yy,xx,cc = 0,0,0
    
    for ii in range(8):

        if (df.columns[ii] == 'VLP_Corr_Name' or df.columns[ii] == 'VLP_Corr' \
            or df.columns[ii] == 'Sol_Gas' or df.columns[ii] == 'Sol_Pressure'):

            col_names_Inputs_L[ii]
        else:    
            AX[xx,yy].hist(df[col_names_Inputs_L[ii]], bins=n_bins, density=True, align='mid', color='orange', alpha=0.7)
            
            AX[xx,yy].set_title(col_names_Inputs_L[ii], weight='bold', fontsize=12)
            AX[xx,yy].set_xlabel(' '),            AX[xx,yy].set_ylabel('')
                        
            AX[xx+1,yy].hist(df_NO[col_names_Inputs_L[ii]], bins=n_bins, density=True, align='mid', color='black', alpha=0.7)
            #AX[xx+1,yy].set_xlabel(' ')
            
            fig.patch.set_linewidth(2),            fig.patch.set_edgecolor('lightgrey')
        
            yy = yy + 1
            cc = cc + 1
    
            if cc > 3:
                cc = 0
                yy = 0
                xx = xx + 2

### Plot Distribution + Cumulative for solutions

In [9]:
def Plot_Dist_Cum_Inputs(df,df_NO,n_bins):
    
    ### -> Create Legend
    fig, ax= plt.subplots(nrows = 1, ncols=1, figsize=(1,1))
    sns.countplot(ax =ax, x=[1], color='red', dodge=False, alpha=0.3)     # -> Label 1    
    sns.countplot(ax =ax, x=[1], color='blue', dodge=False, alpha=0.3)    # -> Label 2
    sns.countplot(ax =ax, x=[1], color='white', dodge=False, alpha=1)
    ax.legend(title='',labels=['Solution', 'No Solution'], facecolor='white', fontsize="15"),    ax.axis('off')
    
    
    
    ### -> Create Plots
    df_col_names_Inputs = df.columns.tolist() ## Get Column names List
    
    fig, AX = plt.subplots(nrows = 4, ncols=4, figsize=(15,10), layout ='constrained')
    
    yy, xx, cc = 0, 0, 0
    #n_bins = 35
    
    for ii in range(8):
      
    
        if (df.columns[ii] == 'VLP_Corr_Name' or df.columns[ii] == 'VLP_Corr' \
            or df.columns[ii] == 'Sol_Gas' or df.columns[ii] == 'Sol_Pressure'):
            if1 = 0
        else:
            #print(df_Input_sol.columns[ii])
            AX[xx,yy].hist(df[df_col_names_Inputs[ii]], bins=n_bins, density=True, align='mid',  \
                           color='red', alpha=0.3, label='Solution', cumulative=False)
    
            AX[xx,yy].set_xlabel(df_col_names_Inputs[ii]),            AX[xx,yy].set_ylabel('PD')
    
            AX[xx,yy].hist(df_NO[df_col_names_Inputs[ii]], bins=n_bins, density=True, align='mid', \
                           color='blue', alpha=0.3, label='NO Solution', cumulative=False)
    
            AX[xx,yy].set_xlabel(''),        AX[xx,yy].set_title(df_col_names_Inputs[ii], weight='bold', fontsize=15)
            
            ### Plot Cumulatives ###
            AX[xx+1,yy].hist(df[df_col_names_Inputs[ii]], bins=n_bins, density=True, align='mid', color='red', alpha=0.3, cumulative=True)
    
            AX[xx+1,yy].set_xlabel(df_col_names_Inputs[ii]),        AX[xx+1,yy].set_ylabel('PD')
    
            AX[xx+1,yy].hist(df_NO[df_col_names_Inputs[ii]], bins=n_bins, density=True, align='mid', color='blue', alpha=0.3, cumulative=True)
    
            AX[xx+1,yy].set_xlabel(' ')#,        AX[xx+1,yy].legend()
            
            yy, cc = yy + 1, cc + 1
    
            if cc > 3:  cc,yy,xx = 0, 0 , xx+2
    
                
    fig.suptitle(' Input data distributions SOLUTION VS NO SOLUTION ', fontsize=20, weight='bold', color='blue');

### Create Cross-Plots for inputs vs Result

In [10]:
def Create_Cross_Plots_Input_vs_Sol(df,Sol_Name, Marker_Color, Marker_Size):

    df_col_names_In = df.columns.tolist()
    print(df_col_names_In)
    
    chart_n = len(df_col_names_In)   
    
    ## check rows needed to plot ##
    if (chart_n/3) > int(chart_n/3):        n_rows = int(chart_n/3) + 1
    if (chart_n/3) == int(chart_n/3):        n_rows = int(chart_n/3) 
    
    ## Create Plots ##
    fig, AX = plt.subplots(nrows = int(n_rows), ncols=3, figsize=(15,10), layout ='constrained')
    
    n, yy, xx, cc = range(chart_n),0,0,0
    
    for ii in n:
        
        if df.columns[ii] != 'VLP_Corr_Name':
            AX[xx,yy].scatter(df[Sol_Name], df[df_col_names_In[ii]], color=Marker_Color, alpha=0.5, s=Marker_Size)
            AX[xx,yy].set_xlabel(Sol_Name),            AX[xx,yy].set_ylabel(df_col_names_In[ii])
    
            # Fit with polyfit - from numpy.polynomial.polynomial import polyfit
            x = df[Sol_Name]
            y = df[df_col_names_In[ii]]
            b, m = np.polynomial.polynomial.polyfit(x,y, 1)
    
    
            AX[xx,yy].plot(x, b + m * x, '-')
    
            yy = yy + 1
            cc = cc + 1
    
            if cc > 2:
                cc = 0
                yy = 0
                xx = xx + 1

#### Plot VLP data for analysis

In [11]:
def VLP_Analysis(VLP_names,VLP_index_list,df_Input_sol,df_Input_All):
    
    VLP_corr_dict = defaultdict(list) # dictionary to save VLP correlations summary
    
    VLP_index_name =[]
    xii  = 0
    for xi in VLP_names:   
        vlp_val = str(VLP_index_list[xii]) + ' ' +  str(VLP_names[xii])
        VLP_index_name.append(vlp_val)
        VLP_count = (df_Input_sol['VLP_Corr'] == VLP_index_list[xii]).sum()
        
        VLP_corr_dict['VLP_Index'].extend([VLP_index_list[xii]])
        VLP_corr_dict['Corr_Name'].extend([str(VLP_names[xii])])
        VLP_corr_dict['VLP_Val_Sum'].extend([VLP_count])
        VLP_corr_dict['Color'].extend([''])
        xii = xii + 1


    df_VLP_corr = pd.DataFrame(VLP_corr_dict) # convert dictionary to DataFrame
    
    g = pd.Categorical(df_VLP_corr.VLP_Index).codes # convert groups to indices  
    n = np.unique(g).size
    palette = sns.color_palette("hls", n).as_hex()
    
    
    display(Markdown(f"## VLP Correlations"))
    # apply the colors to style for VLP Correlation
    display(df_VLP_corr.style.apply(lambda x: ['background-color: {}'.format(palette[i]) for i in g],subset=['Color']))
        
    x,y,y2,xc = df_Input_sol['Sol_Gas'], df_Input_sol['FBHP'], df_Input_sol['Sol_WHP'], df_Input_sol['VLP_Corr']

    fig, ax = plt.subplots(nrows = 5, ncols=1, figsize=(10,10), layout ='constrained')
    
    Pal = "hls"
    
    # --- Plot 0 --- Count values on each VLP correlation
    nplot = ax[0]
    
    s1 = df_Input_All.pivot_table(columns=['VLP_Corr'], aggfunc='size')
    s2 = df_Input_sol.pivot_table(columns=['VLP_Corr'], aggfunc='size')
    
    sns.barplot(ax =nplot, x=s1.index, y=s1.values, color= 'black', alpha=1, dodge=False)
    sns.barplot(ax =nplot, x=s2.index, y=s2.values, palette = Pal, hue =s2.index, dodge=False)
    nplot.legend_.remove()
    
    nplot.set_title('Correlation Counts', weight='bold', color='black')
    nplot.set_xlabel('VLP Correlation Index')
    nplot.set_ylabel('Iteration Counts')
    
    # --- Plot 1 ---
    sns.set_style("white")
    
    nplot = ax[1]
    sns.scatterplot(ax=nplot, x = x, y = y, alpha=0.95, s = 60, hue = xc, palette=Pal, linewidth=0, edgecolor='black', legend=False)
    nplot.set_title('FBHP vs Gas Rate', weight='bold', color='black')
    nplot.set_xlabel('Gas Rate, MMscfd')
    nplot.set_ylabel('FBHP, psi')
    nplot.set_xlim(left=0)
    nplot.grid(color = 'black', linestyle = '--', linewidth = 0.1)
    
    # --- Plot 2 ---
    nplot = ax[2]
    sns.scatterplot(ax=nplot, x = x, y = y2, alpha=0.95, s = 60, hue = xc, palette=Pal, linewidth=0, edgecolor='black', legend=False)
    nplot.set_title('WHP vs Gas Rate', weight='bold', color='black')
    xLabel = 'Gas Rate, MMscfd', '\n'
    nplot.set_xlabel('Gas Rate, MMscfd')
    nplot.set_ylabel('WHP, psi')
    nplot.set_xlim(left=0)
    nplot.grid(color = 'black', linestyle = '--', linewidth = 0.1)
    
    # --- Plot 3 --- Violin Plot for each VLP correlation vs Gas Rate Solution
    nplot = ax[3]
    sns.violinplot(ax = nplot, x = xc, y = x, palette=Pal, legend=False, hue=xc)
    nplot.set_title('VLP Correations', weight='bold', color='black')
    nplot.set_ylabel('Gas Rate')
    nplot.set_xlabel('VLP Correlation Index');
    
    
    # --- Plot 4 --- Violin Plot for each VLP correlation vs BHFP
    nplot = ax[4]
    sns.violinplot(ax = nplot, x = xc, y = y, palette=Pal, legend=False, hue=xc)
    nplot.set_title('VLP Correations', weight='bold', color='black')
    nplot.set_ylabel('BHFP')
    nplot.set_xlabel('VLP Correlation Index');

### Calculate correlation coefficient between variables

In [ ]:
def Calc_Correlation_coeff(df_Input_All, Var_Solution):
    Corr_DF = df_Input_All.copy(deep=True)
    #Corr_DF.drop(['VLP_Corr_Name'], axis=1, inplace=True)  # -> delete column = Correlation name (string variable)
    #display(Corr_DF.head(3))
    Corr_DF = Corr_DF.drop(columns=['VLP_Corr_Name'])
    
    ## -> Check new DataFrame statistics
    #display(Corr_DF.head(3))
    #display(Corr_DF.describe().round(2))

    Corr_DF_Coeff = Corr_DF.corr() ## -> Calculate correlation parameters PANDAS
    
    Corr_DF_Coeff = Corr_DF_Coeff.drop(index=['Sol_Gas', 'Sol_WHP','Sol_Water','FBHP'])
    Corr_DF_Coeff = Corr_DF_Coeff.drop(columns=['Pr','DA','k','h','Perf', 'S', 'WGR','Top_Press','VLP_Corr','Choke'], errors='ignore')
    
    display(Markdown(f"## Correlation Matrix"))
    display(Corr_DF_Coeff.round(3))

    fig, ax = plt.subplots(nrows = 1, ncols=1, figsize=(10,3), layout ='constrained')
    
    sns.barplot(y=Corr_DF_Coeff.index, x=Corr_DF_Coeff[Var_Solution], ax=ax, orient="h", palette='pastel', edgecolor='gray', hue=Corr_DF_Coeff['Sol_Gas'])
    ax.get_legend().remove()

#### Generate Density plots for each VLP correlation specifying X and Y variables

In [12]:
def Density_Plot_VLP_corr(Y_var_str, X_var_str, VLP_index_list,VLP_names,df_Input_sol,df_PDP_hist):
    
    count_corr = len(VLP_index_list)/4
    max_rows = math.ceil(count_corr)
    N_plots = range(max_rows*4) # total number of plots
    
    fig, ax = plt.subplots(nrows=max_rows, ncols=4, figsize=(10,7))
    fig.subplots_adjust(wspace=0.02,hspace=0.2)
    
    max_WHP,max_Qg = df_Input_sol[Y_var_str].max(),  df_Input_sol[X_var_str].max()
    
    pcol, prow, corrL = 0, 0, 0   
    
    for plot_i in N_plots:
        
        if len(VLP_index_list) > plot_i:
            xi = VLP_index_list[plot_i]
    
            df1 = df_Input_sol.loc[df_Input_sol['VLP_Corr'] == xi ]
    
            sns.kdeplot(data=df1, x=X_var_str, y=Y_var_str, levels=6, fill=True, alpha=0.4, cut=2, ax=ax[prow,pcol], 
                        label='PROSPER density map');
    
            if df_PDP_hist.empty == False:
                ax[prow,pcol].scatter(data= df_PDP_hist, x='Gas', y='WHP', color='gray',edgecolors= "black",linewidth=0.5, label='Production Test')
    
            nPlot = ax[prow,pcol]
    
            nPlot.set_title(VLP_names[corrL], weight='bold', fontsize=10)
            nPlot.set_ylabel(Y_var_str, fontsize=9),            nPlot.set_xlabel(X_var_str, fontsize=9)
            nPlot.tick_params(axis='both', which='major', labelsize=8)
            nPlot.set_xlim(left=0, right = max_Qg),            nPlot.set_ylim(bottom=0, top =max_WHP)
            nPlot.yaxis.tick_left(),            nPlot.xaxis.tick_bottom()
    
            if pcol > 0:  nPlot.get_yaxis().set_visible(False)
            if prow < max_rows-1:  nPlot.get_xaxis().set_visible(False)
    
            pcol = pcol + 1
            corrL = corrL+1
            if pcol >3:
                pcol = 0
                prow = prow + 1
        else:
            nPlot = ax[prow,pcol]
            nPlot.get_yaxis().set_visible(False),            nPlot.get_xaxis().set_visible(False)

    fig.suptitle(' PROSPER results (density map) vs Production Test data ', fontsize=15, weight='bold', color='blue')


### Create plot VLP vs IPR, Percentile solutions and distributions

In [13]:
def Create_Plot_VLP_vs_IPR(df_VLP_IPR, df_Input_sol, X_max):

    df = df_VLP_IPR.copy()
    df.loc[df['VLP_Pressure'] == 0, 'VLP_Pressure'] = np.nan # -> replace zero values with NaN on VLP column
    df.loc[df['VLP_Pressure'] > df.IPR_Pressure.max()+50, 'VLP_Pressure'] = np.nan # -> replace zero values with NaN on VLP column
    
    fig = plt.figure(figsize=(15,6))
    gs = gridspec.GridSpec(3, 5) # -> create grid, Rows = 3, Col = 5
    
    fig.subplots_adjust(wspace=0.02)
    fig.subplots_adjust(hspace=0.02)
    
    ax_main = plt.subplot(gs[1:3, :2]) # -> Specify grids associated with Axes
    
    ax_A = plt.subplot(gs[0, :2],sharex=ax_main)                       # -> Rows = 1, Cols = 1,2
    ax_B = plt.subplot(gs[1:3, 3:5],sharey=ax_main)                    # -> Rows = 2,3, Cols = 4,5
    ax_C = plt.subplot(gs[1:3, 2:3],sharey=ax_main)                    # -> Rows = 2,3, Cols = 3,4
    
    
    ## Y vs X plot
    ax = ax_main
    
    i = 0
    while not i > df.Iteration.count():
        df_i = df[df['Iteration'] == i] # -> filter DataFrame based on iteration
        
        #ax.scatter(df_i['Gas Rate'], df_i['VLP_Pressure'], c='darkgray', label='f', alpha=0.5)
        ax.plot(df_i['Gas Rate'], df_i['IPR_Pressure'], c='darkred', label='f', alpha=0.5)
        i = i + 20
        
    #ax.plot(df['Gas Rate'],df['IPR_Pressure'], color=(255/255, 140/255, 0/255), alpha=0.5, clip_on=False)
    ax.scatter(df['Gas Rate'],df['VLP_Pressure'],marker='.', color='darkgray', s= 200, alpha=0.5, clip_on=False)
    
    
    x = np.percentile(df_Input_sol['Sol_Gas'], 90) 
    y = np.percentile(df_Input_sol['FBHP'], 90)
    ax.plot(x, y, label='Solution', marker = 'o', markersize = 12,linestyle= '', color='black', markerfacecolor='red' )
    
    x = np.percentile(df_Input_sol['Sol_Gas'], 10)
    y = np.percentile(df_Input_sol['FBHP'], 10)
    ax.plot(x, y, label='Solution', marker = 'o', markersize = 12,linestyle= '', color='black', markerfacecolor='red' )
    
    x = np.percentile(df_Input_sol['Sol_Gas'], 50)
    y = np.percentile(df_Input_sol['FBHP'], 50)
    ax.plot(x, y, label='Solution', marker = 'o', markersize = 12,linestyle= '', color='black', markerfacecolor='red' )
    
    
    ax.set_xlabel('Gas Rate, MMscfd', color='black', weight='bold')
    ax.set_ylabel('Pressure, psi', color='black', weight='bold')
    
    ax.set_axisbelow(True)
    
    ax.set_xlim(left=0, right = X_max)
    ax.set_ylim(top=df.IPR_Pressure.max()+50,bottom=0)

    
    ax.yaxis.tick_left()
    ax.xaxis.tick_bottom()
    ax.tick_params(axis='x', labelsize=9)
    ax.tick_params(axis='y', labelsize=9)
    
    ##################################
    ### -> Density Map Y vs X plot ###
    ax = ax_B
    contour_set = sns.kdeplot(x=df_Input_sol['Sol_Gas'], y=df_Input_sol['FBHP'], fill=True, alpha=1, ax=ax, cmap='coolwarm', \
                              label='PROSPER density map', thresh=0)
    
    ax.set_xlabel('Gas Rate, MMscfd', color='black', weight='bold')
    ax.tick_params(axis='x', labelsize=8)
    ax.invert_xaxis(), ax.yaxis.tick_right(), ax.invert_xaxis()
    ax.get_yaxis().set_visible(True)
    
    # The last contour is the outermost one
    last_contour_color = contour_set.collections[-1].get_facecolor()[0]
    ax.set_facecolor(last_contour_color)
    
    ############################
    ## -> X axis Distribution ##
    ax = ax_A
    ax.hist(x=df_Input_sol['Sol_Gas'],bins=20,align='mid', color='red', alpha=0.5)
    ax.set(ylabel='PD'), ax.tick_params(axis='x', labelsize=8), ax.tick_params(axis='y', labelsize=8)
    ax.yaxis.tick_left(), ax.xaxis.tick_top()
    
    ## -> Y axis Distribution ##
    ax = ax_C
    ax.hist(x=df_Input_sol['FBHP'],bins=30,orientation='horizontal',align='mid', color='green', alpha=0.4)
    ax.set(xlabel='PD'), ax.tick_params(axis='x', labelsize=8),ax.tick_params(axis='y', labelsize=8)
    
    ax.invert_xaxis(), ax.yaxis.tick_right(), ax.invert_xaxis()
    ax.get_yaxis().set_visible(False)
    plt.show()

### Function to display infor, Head data and Statistics

In [14]:
def Display_Info_Head_Stats_DF(Tittle, df,n_head):

    display(Markdown(f"## {Tittle}"))
    df.info()
    display(df.head(n_head+1).round(2))

    CL1 = df.columns.tolist()    
    
    df1 = pd.DataFrame(df.describe(percentiles=[0.1,.2, .4, .6, .8,0.9]), columns=CL1)
    
    display(Markdown(f"### Statistics "))
    display(df1.round(2))